# Bag-of-Words GRPO

Sample script for zero-step-sampling GRPO on the bag-of-words dataset.

The policy is a Gaussian around the model's scalar prediction,
$m_\theta(z\mid x) = \mathcal N(f_\theta(x), \sigma^2)$. Rollouts are samples from this policy; rewards are $-(z-y)^2$ against the noisy target; advantages are standardized within each rollout group.

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.grpo import BagOfWordsGRPOConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:1")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/grpo.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsGRPOConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-grpo-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
    num_rollouts_per_sample=4,
    gaussian_stdev=1.0,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")
print(f"Rollouts/sample: {config.num_rollouts_per_sample}")
print(f"Policy stdev:    {config.gaussian_stdev}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-grpo-example/7-words_corr-0.2_len-128_pow-1.0_ar-0.5
Dataset corr target: 0.2000
Backbone lr: 1.230e-03
Head lr:     8.192e-03
Rollouts/sample: 4
Policy stdev:    1.0


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
state.run_training()

grpo epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

## Results

Training saves:
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics.
2. Validation parquet containing per-row ground-truth and target.

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

FileNotFoundError: No such file or directory (os error 2): ...stics/artifacts/bow-grpo-example/7-words_corr-0.2_len-128_pow-1.0_ar-0.5/metrics.parquet (set POLARS_VERBOSE=1 to see full path)

In [ ]:
analysis = BagOfWordsAnalysisConfig.from_studies({"example": config.study_folder})
epoch_axis = pl.col("epoch").alias("epoch")

analysis.xy_plots([
    (epoch_axis, rsq_expr(split="train", y="ground_truth"), None),
    (epoch_axis, rsq_expr(split="val", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="train", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="val", y="ground_truth"), None),
])

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()